# El viaje del optimizador

**Explorador de Hespérides · Capítulo 2**

Ampliación programada sobre los conceptos de los notebooks D2L de este capítulo.

La curvatura es quince veces mayor en el segundo eje. Prueba tasas pequeñas y grandes y avanza paso a paso. Los parámetros se mantienen en un recuadro fijo: una trayectoria que sale de él puede estar divergiendo; el gráfico de pérdida lo confirma. Esta función no contiene ruido de minibatch.

![Ilustración conceptual](../recursos/ilustraciones/capitulo_2.png)

*Ilustración conceptual generada con ImageGen. Los resultados cuantitativos son los del código.*

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact

torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(2)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "animation.embed_limit": 40})


In [ ]:

# Misma función y mismo punto inicial para comparar las trayectorias.
def trayectoria(metodo, tasa, pasos=80):
    x = torch.tensor([-3., 2.], requires_grad=True)
    clase = {'SGD': torch.optim.SGD, 'Momentum': torch.optim.SGD, 'Adam': torch.optim.Adam}[metodo]
    extra = {'momentum': .85} if metodo == 'Momentum' else {}
    opt = clase([x], lr=tasa, **extra)
    puntos = [x.detach().numpy().copy()]
    for _ in range(pasos):
        opt.zero_grad()
        perdida = .5 * (x[0]**2 + 15*x[1]**2)
        perdida.backward(); opt.step()
        puntos.append(x.detach().numpy().copy())
        if not torch.isfinite(x).all() or x.abs().max() > 1e5:break
    return np.array(puntos)

def ver_optimizacion(tasa=.08, paso=30):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    xx, yy = np.meshgrid(np.linspace(-4, 4, 150), np.linspace(-3, 3, 150))
    axes[0].contour(xx, yy, .5*(xx**2+15*yy**2), levels=[.1,.5,1,2,4,8,16,32,64], colors='#cccccc')
    for metodo, color in zip(['SGD','Momentum','Adam'], ['#087E8B','#D99B18','#775DA6']):
        p = trayectoria(metodo, tasa)
        k = min(paso, len(p)-1)
        axes[0].plot(*p[:k+1].T, '.-', color=color, label=metodo)
        axes[0].scatter(*p[k], s=65, color=color)
        axes[1].semilogy(.5*(p[:,0]**2+15*p[:,1]**2)+1e-12, color=color, label=metodo)
    axes[0].set(xlim=(-4,4), ylim=(-3,3), xlabel='Parámetro 1', ylabel='Parámetro 2', title='Trayectorias calculadas')
    axes[1].axvline(paso, color='black', alpha=.3)
    axes[1].set(xlabel='Actualización', ylabel='Pérdida (escala logarítmica)', title='Convergencia o divergencia')
    axes[0].legend();fig.tight_layout();plt.show()

interact(ver_optimizacion, tasa=widgets.FloatLogSlider(value=.08, base=10, min=-3, max=-.5, step=.05,
                                                      description='Tasa', continuous_update=False),
         paso=widgets.IntSlider(value=30,min=0,max=80,description='Paso',continuous_update=False));


## Vista de referencia

Esta figura conserva el estado inicial también en una exportación sin kernel. Los controles anteriores se utilizan en Jupyter.

In [ ]:
ver_optimizacion(.08,30)

## Comprobación

Modifica un control cada vez y describe qué cambia y qué permanece constante. Compara tu observación con las preguntas del capítulo.